# W8 Results Evaluation (molecule, facts_50k + paper, predicates + multi-signal scoring)

W8 = predicates (facts + paper) + J (join) + multi-signal weighted sum scoring
(mol jaccard + abstract L2 + join dist).

**Recall**: per query, count baseline rows with `score ≤ worst_GT_score`. `recall = hits / K`.

**Breakdowns**: similarity tier (τ), correlation tier, predicate bucket (`bucket_left × bucket_right`),
selectivity (`facts_sel`, `paper_sel`, `sel_join`), and `expected_candidate_tuples` (post-predicate cart).

In [1]:
import json
from pathlib import Path

RESULTS_DIR = Path('.')
GT_FILE = 'w8_queries_100_gt.json'
WORKLOAD_PATH = Path('..') / '..' / 'workload' / 'w8' / 'w8_queries_100.json'

def load_results(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def gt_answer_to_scores(answer):
    return [(row['fact_id'], row['score']) for row in answer]

def baseline_answer_to_scores(answer):
    # W8 baseline rows: [fact_id, pmid, ..., score] — score is last
    return [(row[0], row[-1]) for row in answer]

def query_results_to_map(data, is_gt=False):
    fn = gt_answer_to_scores if is_gt else baseline_answer_to_scores
    return {r['query_id']: fn(r['answer']) for r in data['results']}

assert (RESULTS_DIR / GT_FILE).exists(), f'Ground truth not found: {GT_FILE}'
gt_data = load_results(RESULTS_DIR / GT_FILE)
gt_by_qid = query_results_to_map(gt_data, is_gt=True)

with open(WORKLOAD_PATH) as f:
    workload = json.load(f)
query_meta = {q['query_id']: q for q in workload['queries']}

BASELINE_FILES = sorted(
    f.name for f in RESULTS_DIR.glob('*.json')
    if f.name != GT_FILE
)

print(f'Ground truth: {GT_FILE}, queries = {len(gt_by_qid)}')
print(f'Baselines ({len(BASELINE_FILES)}):')
for f in BASELINE_FILES:
    print(f'  {f}')

Ground truth: w8_queries_100_gt.json, queries = 100
Baselines (5):
  20260422_221044.json
  20260422_230138.json
  20260422_233904.json
  20260422_234139.json
  20260422_234330.json


In [2]:
def eval_one_baseline(baseline_by_qid, gt_by_qid):
    per_query, total_hits, total_denom = [], 0, 0
    for qid, answers in baseline_by_qid.items():
        if qid not in gt_by_qid:
            continue
        gt_answers = gt_by_qid[qid]
        n = len(gt_answers)
        if n == 0:
            continue
        an = max(score for _, score in gt_answers)
        m = sum(1 for _, score in answers if score <= an + 1e-6)
        per_query.append({'query_id': qid, 'K': n, 'hits': m, 'recall': m / n})
        total_hits += m
        total_denom += n
    mean_recall = total_hits / total_denom if total_denom else 0.0
    return per_query, mean_recall

In [3]:
summary = []
per_query_by_baseline = {}
for f in BASELINE_FILES:
    data = load_results(RESULTS_DIR / f)
    by_qid = query_results_to_map(data, is_gt=False)
    per_query, mean_recall = eval_one_baseline(by_qid, gt_by_qid)
    name = Path(f).stem
    total_sec = data.get('total_elapsed_sec', None)
    n_queries = len(per_query)
    qps = n_queries / total_sec if total_sec and total_sec > 0 else 0.0
    summary.append({
        'baseline':    name,
        'mean_recall': mean_recall,
        'n_queries':   n_queries,
        'qps':         qps,
    })
    per_query_by_baseline[name] = per_query
    print(f'{name}: mean_recall = {mean_recall:.4f}, qps = {qps:.3f}')

20260422_221044: mean_recall = 0.9975, qps = 0.579
20260422_230138: mean_recall = 0.9975, qps = 0.727
20260422_233904: mean_recall = 0.9915, qps = 0.769
20260422_234139: mean_recall = 0.9915, qps = 0.932
20260422_234330: mean_recall = 0.9915, qps = 0.852


In [4]:
import pandas as pd

df = pd.DataFrame(summary).sort_values("mean_recall", ascending=False)
print(df.to_string(index=False))

       baseline  mean_recall  n_queries      qps
20260422_221044       0.9975        100 0.578788
20260422_230138       0.9975        100 0.726907
20260422_233904       0.9915        100 0.768591
20260422_234139       0.9915        100 0.931714
20260422_234330       0.9915        100 0.852001
